# Reproducing and Extending SimCSE
COMP8240 Applications of Data Science, Macquarie University, Session 2 2026 — Trilok Kumar Reddy Ande

Run the cells from top to bottom on a GPU runtime (Google Colab, T4): Runtime → Run all. The first code cell copies the saved WikiSim data from this repository, so the same sentences and pairs are rebuilt. See `README.md` for details.

## 0. Setup

In [1]:
# Paths: everything is saved in a local "data" folder next to this notebook.
# Set USE_DRIVE = True to save to Google Drive instead (Colab only).
import os
USE_DRIVE = False
ROOT = os.getcwd()
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    SAVE_DIR = "/content/drive/MyDrive/COMP8240_WikiSim"
else:
    SAVE_DIR = os.path.join(ROOT, "data")
os.makedirs(SAVE_DIR, exist_ok=True)

# If the saved WikiSim data is not here yet (e.g. the notebook was opened
# directly in Colab), copy it from this GitHub repository's data/ folder.
# Using raw_articles.json keeps the exact same Wikipedia revisions.
if not USE_DRIVE and not os.path.exists(os.path.join(SAVE_DIR, "raw_articles.json")):
    os.system("rm -rf /tmp/repo && git clone -q https://github.com/Trilok119/comp8240-simcse-replication /tmp/repo")
    os.system(f"cp -r /tmp/repo/data/. {SAVE_DIR}/")
print("Saving data to", SAVE_DIR)
print("Files:", sorted(os.listdir(SAVE_DIR)))

Saving data to /content/data
Files: ['annotated_100.csv', 'annotation_claude.csv', 'annotation_done.csv', 'annotation_sheet.csv', 'raw_articles.json', 'sentences.json', 'wikisim_pairs.csv']


In [2]:
import sys, torch, transformers
print("Python      :", sys.version.split()[0])
print("torch       :", torch.__version__)
print("transformers:", transformers.__version__)
print("GPU         :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

Python      : 3.13.16
torch       : 2.14.1+cu130
transformers: 5.22.1
GPU         : Tesla T4


## 1. Replication of the original results
Clone the official SimCSE code. The `sed` line fixes `batch_encode_plus`, which was removed from newer versions of `transformers`.

In [3]:
!git clone https://github.com/princeton-nlp/SimCSE.git
%cd SimCSE
!pip install -q transformers scipy scikit-learn prettytable
!sed -i "s/tokenizer.batch_encode_plus(/tokenizer(/g" evaluation.py

Cloning into 'SimCSE'...
remote: Enumerating objects: 453, done.
remote: Counting objects: 100% (167/167), done.
remote: Compressing objects: 100% (59/59), done.
remote: Total 453 (delta 123), reused 108 (delta 108), pack-reused 286 (from 1)
Receiving objects: 100% (453/453), 41.46 MiB | 20.81 MiB/s, done.
Resolving deltas: 100% (229/229), done.
/content/SimCSE


In [4]:
%cd SentEval/data/downstream/
!bash download_dataset.sh
%cd /content/SimCSE

/content/SimCSE/SentEval/data/downstream
--2026-11-07 03:00:21--  https://huggingface.co/datasets/princeton-nlp/datasets-for-simcse/resolve/main/senteval.tar
Resolving huggingface.co (huggingface.co)... 18.164.174.23, 18.164.174.55, 18.164.174.17, ...
Connecting to huggingface.co (huggingface.co)|18.164.174.23|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/621ffdd236468d709f183d48/88d56ab28dbc96b9b8895a39ecdcf8211ed8a2942ee7bc33b79421ff27c5a841?X-Xet-Cas-Uid=public&response-content-type=application%2Fx-tar&user_id=public&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27senteval.tar%3B+filename%3D%22senteval.tar%22%3B&xip=lFctjkq_Eno&Expires=1794024021&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjIxZmZkZDIzNjQ2OGQ3MDlmMTgzZDQ4Lzg4ZDU2YWIyOGRiYzk2YjliODg5NWEzOWVjZGNmODIxMWVkOGEyOTQyZWU3YmMzM2I3OTQyMWZmMjdjNWE4NDFcXD9YLVhldC1DYXMtVWlkPXB1YmxpYyZyZXNwb25zZ

### 1.1 Supervised SimCSE (BERT-base)

In [5]:
!python evaluation.py \
  --model_name_or_path princeton-nlp/sup-simcse-bert-base-uncased \
  --pooler cls --task_set sts --mode test

/content/SimCSE/SentEval/senteval/utils.py:62: SyntaxWarning: invalid escape sequence '\d'
  assert re.match("^[+-]?(\d+(\.\d*)?|\.\d+)$", split[1]) is not None
2026-11-07 03:00:35,689 : NumExpr defaulting to 2 threads.
2026-11-07 03:00:37,649 : connect_tcp.started host='huggingface.co' port=443 local_address=None timeout=3 socket_options=None
2026-11-07 03:00:37,665 : connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x78d7037412b0>
2026-11-07 03:00:37,665 : start_tls.started ssl_context=<ssl.SSLContext object at 0x78d7038051d0> server_hostname='huggingface.co' timeout=3
2026-11-07 03:00:37,677 : start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x78d703744190>
2026-11-07 03:00:37,679 : send_request_headers.started request=<Request [b'GET']>
2026-11-07 03:00:37,679 : send_request_headers.complete
2026-11-07 03:00:37,679 : send_request_body.started request=<Request [b'GET']>
2026-11-07 03:00:37,680 : send_request_body.complete
2

### 1.2 Unsupervised SimCSE (BERT-base)

In [6]:
!python evaluation.py \
  --model_name_or_path princeton-nlp/unsup-simcse-bert-base-uncased \
  --pooler cls_before_pooler --task_set sts --mode test

2026-11-07 03:02:43,337 : NumExpr defaulting to 2 threads.
2026-11-07 03:02:43,722 : connect_tcp.started host='huggingface.co' port=443 local_address=None timeout=None socket_options=None
2026-11-07 03:02:43,734 : connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7e8b0e1002f0>
2026-11-07 03:02:43,734 : start_tls.started ssl_context=<ssl.SSLContext object at 0x7e8b0e0d9450> server_hostname='huggingface.co' timeout=None
2026-11-07 03:02:43,744 : start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7e8b0e0e4cd0>
2026-11-07 03:02:43,744 : send_request_headers.started request=<Request [b'GET']>
2026-11-07 03:02:43,745 : send_request_headers.complete
2026-11-07 03:02:43,745 : send_request_body.started request=<Request [b'GET']>
2026-11-07 03:02:43,745 : send_request_body.complete
2026-11-07 03:02:43,745 : receive_response_headers.started request=<Request [b'GET']>
2026-11-07 03:02:43,834 : receive_response_headers.complete return_val

### 1.3 Sanity check

In [7]:
from simcse import SimCSE

model = SimCSE("princeton-nlp/sup-simcse-bert-base-uncased")

query = ["Two dogs are running."]
others = [
    "There are animals outdoors.",       # should be similar (entailment)
    "The pets are sitting on a couch.",  # should be less similar (contradiction)
    "The stock market fell today.",      # should be very different (unrelated)
]

scores = model.similarity(query, others)[0]
for sentence, score in zip(others, scores):
    print(f"{score:.3f}  {sentence}")

print()
print("Tricky pair (same words, different meaning):")
print(f"{model.similarity('The dog bit the man.', 'The man bit the dog.'):.3f}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00, 12.77it/s]


0.736  There are animals outdoors.
0.278  The pets are sitting on a couch.
0.026  The stock market fell today.

Tricky pair (same words, different meaning):


100%|██████████| 1/1 [00:00<00:00, 109.81it/s]

0.951


## 2. New existing data: PAWS

In [8]:
!pip install -q datasets
from datasets import load_dataset

paws = load_dataset("google-research-datasets/paws", "labeled_final", split="test")

print("Number of pairs:", len(paws))
print("Columns:", paws.column_names)
print("Label counts:", {0: paws["label"].count(0), 1: paws["label"].count(1)})
print()
for i in range(3):
    print("Sentence 1:", paws[i]["sentence1"])
    print("Sentence 2:", paws[i]["sentence2"])
    print("Label:", paws[i]["label"], "(1 = same meaning, 0 = different meaning)")
    print()

README.md:   0%|          | 0.00/9.79k [00:00<?, ?B/s]

labeled_final/train-00000-of-00001.parqu(…): reconstructing file:   0%|          |  0.00B / 8.43MB            

labeled_final/train-00000-of-00001.parqu(…): downloading bytes:           |  0.00B            

labeled_final/test-00000-of-00001.parque(…): reconstructing file:   0%|          |  0.00B / 1.24MB            

labeled_final/test-00000-of-00001.parque(…): downloading bytes:           |  0.00B            

labeled_final/validation-00000-of-00001.(…): reconstructing file:   0%|          |  0.00B / 1.23MB            

labeled_final/validation-00000-of-00001.(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/49401 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/8000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/8000 [00:00<?, ? examples/s]

Number of pairs: 8000
Columns: ['id', 'sentence1', 'sentence2', 'label']
Label counts: {0: 4464, 1: 3536}

Sentence 1: This was a series of nested angular standards , so that measurements in azimuth and elevation could be done directly in polar coordinates relative to the ecliptic .
Sentence 2: This was a series of nested polar scales , so that measurements in azimuth and elevation could be performed directly in angular coordinates relative to the ecliptic .
Label: 0 (1 = same meaning, 0 = different meaning)

Sentence 1: His father emigrated to Missouri in 1868 but returned when his wife became ill and before the rest of the family could also go to America .
Sentence 2: His father emigrated to America in 1868 , but returned when his wife became ill and before the rest of the family could go to Missouri .
Label: 0 (1 = same meaning, 0 = different meaning)

Sentence 1: In January 2011 , the Deputy Secretary General of FIBA Asia , Hagop Khajirian , inspected the venue together with SBP - 

In [9]:
from sklearn.metrics import roc_auc_score
import numpy as np

s1 = paws["sentence1"]
s2 = paws["sentence2"]
labels = np.array(paws["label"])

def simcse_scores(name):
    m = SimCSE(name)
    a = m.encode(s1, batch_size=128)
    b = m.encode(s2, batch_size=128)
    return (a * b).sum(dim=1).cpu().numpy()   # cosine similarity for each pair

def overlap_scores():
    out = []
    for x, y in zip(s1, s2):
        A, B = set(x.lower().split()), set(y.lower().split())
        out.append(len(A & B) / len(A | B))     # share of words in common
    return np.array(out)

paws_results = {
    "Word overlap (baseline)": overlap_scores(),
    "Unsup SimCSE": simcse_scores("princeton-nlp/unsup-simcse-bert-base-uncased"),
    "Sup SimCSE": simcse_scores("princeton-nlp/sup-simcse-bert-base-uncased"),
}

print()
for name, sc in paws_results.items():
    auc = roc_auc_score(labels, sc)
    print(f"{name:25s} ROC-AUC = {auc:.3f} | avg score: same meaning = {sc[labels==1].mean():.3f}, different = {sc[labels==0].mean():.3f}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

100%|██████████| 63/63 [00:25<00:00,  2.46it/s]


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

100%|██████████| 63/63 [00:22<00:00,  2.75it/s]



Word overlap (baseline)   ROC-AUC = 0.498 | avg score: same meaning = 0.888, different = 0.887
Unsup SimCSE              ROC-AUC = 0.585 | avg score: same meaning = 0.975, different = 0.969
Sup SimCSE                ROC-AUC = 0.647 | avg score: same meaning = 0.979, different = 0.970


### 2.1 Highest-scoring different-meaning pairs

In [10]:
sc = paws_results["Sup SimCSE"]
different = np.where(labels == 0)[0]                     # only different-meaning pairs
worst = different[np.argsort(-sc[different])[:3]]        # the 3 highest scores among them

for i in worst:
    print(f"Score: {sc[i]:.3f}   (true label: 0 = different meaning)")
    print("  1:", s1[i])
    print("  2:", s2[i])
    print()

Score: 1.000   (true label: 0 = different meaning)
  1: This is a list of international sporting events in the Greater Manila Area in the Philippines :
  2: This is a list of international sporting events in the Greater Manila Area in the Philippines :

Score: 1.000   (true label: 0 = different meaning)
  1: The estimated middle income for a household in the city in 2013 was $ 63,121 , while the median income for a household in the city was $ 78,090 .
  2: The estimated middle income for a household in the city in 2013 was $ 63,121 , while the median income for a household in the city was $ 78,090 .

Score: 1.000   (true label: 0 = different meaning)
  1: To read about the background to these events , see the history of Malaysia .
  2: To read about the background to these events , see the history of Malaysia .



### 2.2 Identical pairs and real mistakes

In [11]:
identical = np.array([a.strip() == b.strip() for a, b in zip(s1, s2)])
print("Identical pairs:", identical.sum(), "out of", len(labels))
print("  labelled 0 (different):", (identical & (labels == 0)).sum())
print("  labelled 1 (same):     ", (identical & (labels == 1)).sum())
print()

different_real = np.where((labels == 0) & ~identical)[0]
worst = different_real[np.argsort(-sc[different_real])[:3]]

print("Worst real mistakes (non-identical pairs):\n")
for i in worst:
    print(f"Score: {sc[i]:.3f}   (true label: 0 = different meaning)")
    print("  1:", s1[i])
    print("  2:", s2[i])
    print()

Identical pairs: 13 out of 8000
  labelled 0 (different): 4
  labelled 1 (same):      9

Worst real mistakes (non-identical pairs):

Score: 1.000   (true label: 0 = different meaning)
  1: She was born in PEI , studied in Halifax and lived in Toronto and has a BA in English from Dalhousie University .
  2: She was born in PEI , studied in Toronto and lived in Halifax and has a BA in English from Dalhousie University .

Score: 1.000   (true label: 0 = different meaning)
  1: The schism was not repaired until the late 6th century for Milan and in the late 7th century for Aquileia .
  2: The schism was not repaired until the late 7th century for Milan and in the late 6th century for Aquileia .

Score: 1.000   (true label: 0 = different meaning)
  1: The schism was not repaired until the late 7th century for Milan and for Aquileia in the late 6th century .
  2: The schism was not repaired until the late 6th century for Milan and for Aquileia in the late 7th century .



## 3. New constructed data: WikiSim
### 3.1 Download 20 Wikipedia articles (with retries and revision IDs)

In [12]:
import requests, json, time

TOPICS = {
    "animals":   ["Koala", "Kangaroo", "Platypus", "Wombat"],
    "planets":   ["Mars", "Jupiter", "Venus", "Saturn"],
    "sports":    ["Cricket", "Rugby union", "Tennis", "Association football"],
    "buildings": ["Sydney Opera House", "Eiffel Tower", "Taj Mahal", "Colosseum"],
    "diseases":  ["Malaria", "Tuberculosis", "Influenza", "Measles"],
}

HEADERS = {"User-Agent": "COMP8240-SimCSE-replication (student project; https://github.com/Trilok119/comp8240-simcse-replication)"}

def fetch(title, tries=5):
    for attempt in range(tries):
        r = requests.get(
            "https://en.wikipedia.org/w/api.php",
            params={"action": "query", "prop": "extracts|revisions", "explaintext": 1,
                    "exsectionformat": "wiki", "rvprop": "ids|timestamp",
                    "titles": title, "redirects": 1, "format": "json"},
            headers=HEADERS,
        )
        try:
            page = next(iter(r.json()["query"]["pages"].values()))
            return page["extract"], page["revisions"][0]["revid"], page["revisions"][0]["timestamp"]
        except Exception:
            wait = 5 * (attempt + 1)
            print(f"   ...empty reply for {title} (status {r.status_code}), waiting {wait}s and retrying")
            time.sleep(wait)
    raise RuntimeError(f"Could not download {title}")

# If raw_articles.json is already in the data folder (it is in the GitHub repo),
# load it so the exact same Wikipedia revisions are used. Otherwise download.
RAW_PATH = f"{SAVE_DIR}/raw_articles.json"
raw = json.load(open(RAW_PATH)) if os.path.exists(RAW_PATH) else {}
print("Articles already saved:", len(raw))

for topic, titles in TOPICS.items():
    for t in titles:
        if t in raw:
            continue                      # already downloaded
        text, revid, ts = fetch(t)
        raw[t] = {"topic": topic, "text": text, "revid": revid, "timestamp": ts}
        print(f"{topic:10s} {t:22s} {len(text.split()):6d} words   (revision {revid})")
        time.sleep(1)                     # be polite to Wikipedia

with open(RAW_PATH, "w") as f:
    json.dump(raw, f)

print("\nSaved", len(raw), "articles to", SAVE_DIR)

Articles already saved: 20

Saved 20 articles to /content/data


### 3.2 Clean and split into sentences

In [13]:
import re, random, nltk
nltk.download("punkt_tab", quiet=True)
from nltk.tokenize import sent_tokenize

SKIP = {"see also", "references", "external links", "further reading", "notes",
        "bibliography", "sources", "citations", "gallery", "footnotes"}

records = []
stats = {"kept": 0, "too_short": 0, "too_long": 0, "junk": 0}

for title, art in raw.items():
    section, para_id = "Introduction", 0
    for line in art["text"].split("\n"):
        line = line.strip()
        if not line:
            continue
        heading = re.match(r"^(=+)\s*(.*?)\s*=+$", line)
        if heading:
            if len(heading.group(1)) == 2:          # main section heading (== Name ==)
                section = heading.group(2)
            continue
        if section.lower() in SKIP:
            continue
        para_id += 1
        for pos, s in enumerate(sent_tokenize(line)):
            n = len(s.split())
            if n < 8:
                stats["too_short"] += 1; continue
            if n > 40:
                stats["too_long"] += 1; continue
            if any(c in s for c in "{}\\") or not re.search("[a-zA-Z]", s):
                stats["junk"] += 1; continue
            records.append({"article": title, "topic": art["topic"], "section": section,
                            "para": para_id, "pos": pos, "text": s})
            stats["kept"] += 1

print("Cleaning summary:", stats)
print("Total clean sentences:", len(records))
print()
for title in raw:
    rows = [r for r in records if r["article"] == title]
    print(f"{title:22s} {len(rows):4d} sentences, {len({r['section'] for r in rows}):2d} sections")

print("\nThree random examples:")
random.seed(42)
for r in random.sample(records, 3):
    print(f"  [{r['topic']} | {r['article']} | {r['section']}] {r['text']}")

with open(f"{SAVE_DIR}/sentences.json", "w") as f:
    json.dump(records, f)
print("\nSaved to", SAVE_DIR)

Cleaning summary: {'kept': 6098, 'too_short': 226, 'too_long': 363, 'junk': 0}
Total clean sentences: 6098

Koala                   295 sentences,  8 sections
Kangaroo                241 sentences,  6 sections
Platypus                203 sentences,  8 sections
Wombat                  111 sentences,  8 sections
Mars                    344 sentences, 11 sections
Jupiter                 333 sentences, 10 sections
Venus                   347 sentences,  8 sections
Saturn                  229 sentences,  8 sections
Cricket                 328 sentences,  8 sections
Rugby union             438 sentences, 14 sections
Tennis                  471 sentences, 13 sections
Association football    298 sentences,  9 sections
Sydney Opera House      258 sentences,  9 sections
Eiffel Tower            263 sentences,  9 sections
Taj Mahal               207 sentences,  9 sections
Colosseum               219 sentences,  8 sections
Malaria                 448 sentences, 15 sections
Tuberculosis            3

### 3.3 Build 400 pairs at four relatedness levels

In [14]:
from collections import defaultdict
import pandas as pd

random.seed(42)
N = 100

# Level 3: neighbouring sentences in the same paragraph
by_para = defaultdict(list)
for i, r in enumerate(records):
    by_para[(r["article"], r["para"])].append(i)
adjacent = [(a, b) for idx in by_para.values() for a, b in zip(idx, idx[1:])
            if records[b]["pos"] - records[a]["pos"] == 1]
level3 = random.sample(adjacent, N)

# Levels 2, 1, 0: random pairs that follow each rule
def pick(rule, n):
    chosen = set()
    while len(chosen) < n:
        a, b = random.sample(range(len(records)), 2)
        if rule(records[a], records[b]):
            chosen.add((min(a, b), max(a, b)))
    return list(chosen)

level2 = pick(lambda x, y: x["article"] == y["article"] and x["section"] != y["section"], N)
level1 = pick(lambda x, y: x["topic"] == y["topic"] and x["article"] != y["article"], N)
level0 = pick(lambda x, y: x["topic"] != y["topic"], N)

rows = []
for level, pairs in [(3, level3), (2, level2), (1, level1), (0, level0)]:
    for a, b in pairs:
        A, B = records[a], records[b]
        rows.append({"level": level, "s1": A["text"], "s2": B["text"],
                     "article1": A["article"], "article2": B["article"],
                     "section1": A["section"], "section2": B["section"]})

df = pd.DataFrame(rows).sample(frac=1, random_state=42).reset_index(drop=True)
df.insert(0, "pair_id", range(1, len(df) + 1))

def overlap(x, y):
    A, B = set(x.lower().split()), set(y.lower().split())
    return len(A & B) / len(A | B)
df["word_overlap"] = [overlap(x, y) for x, y in zip(df.s1, df.s2)]

df.to_csv(f"{SAVE_DIR}/wikisim_pairs.csv", index=False)

print("Pairs per level:", df.level.value_counts().sort_index().to_dict())
print("Neighbouring-sentence candidates available:", len(adjacent))
print("\nAverage word overlap per level:")
print(df.groupby("level").word_overlap.mean().round(3).to_dict())

print("\nOne example per level:")
for lv in [3, 2, 1, 0]:
    r = df[df.level == lv].iloc[0]
    print(f"\nLevel {lv}: [{r.article1} / {r.section1}]  vs  [{r.article2} / {r.section2}]")
    print("  1:", r.s1)
    print("  2:", r.s2)

print("\nSaved", len(df), "pairs to", SAVE_DIR)

Pairs per level: {0: 100, 1: 100, 2: 100, 3: 100}
Neighbouring-sentence candidates available: 4155

Average word overlap per level:
{0: 0.049, 1: 0.054, 2: 0.06, 3: 0.093}

One example per level:

Level 3: [Taj Mahal / Tourism]  vs  [Taj Mahal / Tourism]
  1: Visitors are allowed through three gates and, as polluting vehicles are not allowed near the complex, tourists must either walk or take electric buses from the designated parking areas.
  2: The complex is open on all days except on Fridays, from one hour before sunrise to 45 minutes before sunset.

Level 2: [Influenza / Signs and symptoms]  vs  [Influenza / Prevention]
  1: The time between exposure to the virus and development of symptoms (the incubation period) is one to four days, most commonly one to two days.
  2: Smoking raises the risk of contracting influenza, as well as producing more severe disease symptoms.

Level 1: [Koala / Taxonomy]  vs  [Kangaroo / Introduction]
  1: Other names suggested by European authors includ

### 3.4 Score WikiSim with SimCSE

In [15]:
from scipy.stats import spearmanr

def encode_pairs(name, a_list, b_list):
    m = SimCSE(name)
    a = m.encode(a_list, batch_size=128)
    b = m.encode(b_list, batch_size=128)
    return (a * b).sum(dim=1).cpu().numpy()

s1_list, s2_list = df.s1.tolist(), df.s2.tolist()
df["unsup_simcse"] = encode_pairs("princeton-nlp/unsup-simcse-bert-base-uncased", s1_list, s2_list)
df["sup_simcse"]   = encode_pairs("princeton-nlp/sup-simcse-bert-base-uncased", s1_list, s2_list)

print()
for col, name in [("word_overlap", "Word overlap (baseline)"),
                  ("unsup_simcse", "Unsup SimCSE"),
                  ("sup_simcse", "Sup SimCSE")]:
    rho = spearmanr(df.level, df[col]).correlation
    print(f"{name:25s} Spearman = {rho * 100:.2f}")

print("\nAverage similarity score per level:")
print(df.groupby("level")[["word_overlap", "unsup_simcse", "sup_simcse"]].mean().round(3))

df.to_csv(f"{SAVE_DIR}/wikisim_pairs.csv", index=False)
print("\nSaved scores to", SAVE_DIR)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

100%|██████████| 4/4 [00:01<00:00,  3.14it/s]


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

100%|██████████| 4/4 [00:01<00:00,  3.24it/s]


Word overlap (baseline)   Spearman = 30.29
Unsup SimCSE              Spearman = 70.13
Sup SimCSE                Spearman = 71.49

Average similarity score per level:
       word_overlap  unsup_simcse  sup_simcse
level                                        
0             0.049         0.269       0.065
1             0.054         0.368       0.266
2             0.060         0.389       0.301
3             0.093         0.551       0.501

Saved scores to /content/data


### 3.5 Create the annotation sheet (100 pairs, levels hidden)

In [16]:
ann_sample = pd.concat([df[df.level == lv].sample(25, random_state=7) for lv in [0, 1, 2, 3]])
ann = ann_sample[["pair_id", "s1", "s2"]].sample(frac=1, random_state=7).reset_index(drop=True)
ann["human_score"] = ""

ann.to_csv(f"{SAVE_DIR}/annotation_sheet.csv", index=False)
print("Created annotation sheet with", len(ann), "pairs")
print("Levels in sample:", ann_sample.level.value_counts().sort_index().to_dict())
print("\nFirst 3 rows:")
print(ann.head(3).to_string())

Created annotation sheet with 100 pairs
Levels in sample: {0: 25, 1: 25, 2: 25, 3: 25}

First 3 rows:
   pair_id                                                                                                                                                                      s1                                                                                                                                                                                                                               s2 human_score
0      230                                   Craters larger than roughly 50 km are named for deceased scientists and writers and others who have contributed to the study of Mars.                                                                                                                     It has been suggested that Venusian lightning (discussed below) could originate from volcanic activity (i.e.            
1      113                                                        

### 3.6 Check for broken sentences

In [17]:
def is_broken(s):
    unbalanced = s.count("(") != s.count(")")
    bad_end = re.search(r"\b(i\.e|e\.g|c|ca|approx|St|Dr|Mt|vs|No)\.$", s) is not None
    return unbalanced or bad_end

broken_sents = [r["text"] for r in records if is_broken(r["text"])]
pair_broken = df.s1.apply(is_broken) | df.s2.apply(is_broken)
ann_broken = ann.s1.apply(is_broken) | ann.s2.apply(is_broken)

print(f"Broken sentences: {len(broken_sents)} of {len(records)} ({100*len(broken_sents)/len(records):.1f}%)")
print(f"Pairs with a broken sentence: {pair_broken.sum()} of {len(df)}")
print(f"Annotation pairs with a broken sentence: {ann_broken.sum()} of {len(ann)}")
print("\nFive examples:")
for s in broken_sents[:5]:
    print("  -", s)

Broken sentences: 35 of 6098 (0.6%)
Pairs with a broken sentence: 6 of 400
Annotation pairs with a broken sentence: 1 of 100

Five examples:
  - These ten chromosomes form five unique pairs of XY in males and XX in females, i.e.
  - Their incisor teeth somewhat resemble those of rodents (rats, mice, etc.
  - In Latin, Iovis is the genitive case of Iuppiter, i.e.
  - A series of global maps of Jupiter's thermospheric temperature, dominant molecular ion (
  - It has been suggested that Venusian lightning (discussed below) could originate from volcanic activity (i.e.


## 4. Human annotation
`annotation_done.csv` holds the author's 0–5 scores for the 100 pairs.

In [18]:
human = pd.read_csv(f"{SAVE_DIR}/annotation_done.csv")[["pair_id", "human_score"]]
merged = df.merge(human, on="pair_id")
print("Annotated pairs found:", len(merged))

print("\n1) Do my scores agree with the automatic levels?")
print(f"   Spearman (human vs level) = {spearmanr(merged.level, merged.human_score).correlation * 100:.2f}")
print("   My average score per level:", merged.groupby("level").human_score.mean().round(2).to_dict())

print("\n2) How well does each method match MY scores?")
for col, name in [("word_overlap", "Word overlap"), ("unsup_simcse", "Unsup SimCSE"), ("sup_simcse", "Sup SimCSE")]:
    vs_human = spearmanr(merged.human_score, merged[col]).correlation * 100
    vs_level = spearmanr(merged.level, merged[col]).correlation * 100
    print(f"   {name:14s} vs my scores = {vs_human:.2f}   |   vs levels (same 100 pairs) = {vs_level:.2f}")

merged.to_csv(f"{SAVE_DIR}/annotated_100.csv", index=False)
print("\nSaved to", SAVE_DIR)

Annotated pairs found: 100

1) Do my scores agree with the automatic levels?
   Spearman (human vs level) = 76.87
   My average score per level: {0: 1.12, 1: 2.0, 2: 4.0, 3: 4.68}

2) How well does each method match MY scores?
   Word overlap   vs my scores = 34.04   |   vs levels (same 100 pairs) = 30.75
   Unsup SimCSE   vs my scores = 67.52   |   vs levels (same 100 pairs) = 72.10
   Sup SimCSE     vs my scores = 67.31   |   vs levels (same 100 pairs) = 68.73

Saved to /content/data


### 4.1 Agreement with a second (AI) annotator

In [19]:
# Agreement between the author's scores and a second annotator.
# NOTE: the second annotator is an AI model (Claude, Anthropic), not a person.
from sklearn.metrics import cohen_kappa_score
a = pd.read_csv(f"{SAVE_DIR}/annotation_done.csv")[["pair_id", "human_score"]].rename(columns={"human_score": "author"})
b = pd.read_csv(f"{SAVE_DIR}/annotation_claude.csv")[["pair_id", "human_score"]].rename(columns={"human_score": "ai_annotator"})
both = a.merge(b, on="pair_id")
print("Pairs:", len(both))
print(f"Spearman                 = {spearmanr(both.author, both.ai_annotator).correlation * 100:.2f}")
print(f"Quadratic weighted kappa = {cohen_kappa_score(both.author, both.ai_annotator, weights='quadratic'):.3f}")
print(f"Exact agreement          = {(both.author == both.ai_annotator).mean():.0%}")
print(f"Within 1 point           = {((both.author - both.ai_annotator).abs() <= 1).mean():.0%}")

Pairs: 100
Spearman                 = 76.79
Quadratic weighted kappa = 0.715
Exact agreement          = 30%
Within 1 point           = 77%


## 5. LLM as a judge (Qwen2.5-3B-Instruct)
### 5.1 Version 1 prompt

In [20]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

LLM_NAME = "Qwen/Qwen2.5-3B-Instruct"
tok = AutoTokenizer.from_pretrained(LLM_NAME)
llm = AutoModelForCausalLM.from_pretrained(LLM_NAME, dtype=torch.float16).to("cuda")

GUIDE = """You rate how related two sentences are, on a scale from 0 to 5.
5 = same specific point
4 = same subject, closely connected details
3 = same subject, different aspect
2 = same broad area, different subject (e.g. koala vs kangaroo)
1 = very loosely connected
0 = unrelated
Reply with a single digit only."""

def judge(a, b):
    msgs = [{"role": "system", "content": GUIDE},
            {"role": "user", "content": f"Sentence 1: {a}\nSentence 2: {b}\nScore:"}]
    prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ids = tok(prompt, return_tensors="pt").to("cuda")
    out = llm.generate(**ids, max_new_tokens=3, do_sample=False)
    reply = tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)
    m = re.search(r"[0-5]", reply)
    return int(m.group()) if m else None

merged["llm_score"] = [judge(a, b) for a, b in zip(merged.s1, merged.s2)]

ok = merged.llm_score.notna()
print("Pairs the AI scored:", ok.sum(), "of", len(merged))
print("AI score counts:", merged.llm_score.value_counts().sort_index().to_dict())
print(f"\nAI vs my scores: Spearman = {spearmanr(merged.human_score[ok], merged.llm_score[ok]).correlation * 100:.2f}")
print(f"AI vs levels:    Spearman = {spearmanr(merged.level[ok], merged.llm_score[ok]).correlation * 100:.2f}")
print("AI average per level:", merged[ok].groupby("level").llm_score.mean().round(2).to_dict())

merged.to_csv(f"{SAVE_DIR}/annotated_100.csv", index=False)
print("\nSaved to", SAVE_DIR)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Pairs the AI scored: 100 of 100
AI score counts: {0: 42, 1: 48, 2: 2, 3: 8}

AI vs my scores: Spearman = 26.59
AI vs levels:    Spearman = 25.06
AI average per level: {0: 0.6, 1: 0.44, 2: 0.68, 3: 1.32}

Saved to /content/data


### 5.2 Version 2 prompt

In [21]:
GUIDE_V2 = """You rate how related two sentences are, on a scale from 0 to 5.
Two sentences can be related even if they share no words: judge whether they are about
the same thing, the same kind of thing, or nothing in common.

5 = same specific point (e.g. both about the opening hours of the same building)
4 = same subject, closely connected details (e.g. a building's entry gates and its opening hours)
3 = same subject, different aspect (e.g. a disease's symptoms and its prevention)
2 = same broad area, different subject (e.g. one about koalas, one about kangaroos)
1 = very loosely connected (e.g. both just mention the same country or a date)
0 = unrelated (e.g. one about kangaroos, one about virus genetics)
Reply with a single digit only."""

def judge_v2(a, b):
    msgs = [{"role": "system", "content": GUIDE_V2},
            {"role": "user", "content": f"Sentence 1: {a}\nSentence 2: {b}\nScore:"}]
    prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ids = tok(prompt, return_tensors="pt").to("cuda")
    out = llm.generate(**ids, max_new_tokens=3, do_sample=False)
    reply = tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)
    m = re.search(r"[0-5]", reply)
    return int(m.group()) if m else None

merged["llm_score_v2"] = [judge_v2(a, b) for a, b in zip(merged.s1, merged.s2)]

ok = merged.llm_score_v2.notna()
print("Pairs scored:", ok.sum(), "of", len(merged))
print("AI score counts (v2):", merged.llm_score_v2.value_counts().sort_index().to_dict())
print(f"\nAI v2 vs my scores: Spearman = {spearmanr(merged.human_score[ok], merged.llm_score_v2[ok]).correlation * 100:.2f}")
print(f"AI v2 vs levels:    Spearman = {spearmanr(merged.level[ok], merged.llm_score_v2[ok]).correlation * 100:.2f}")
print("AI v2 average per level:", merged[ok].groupby("level").llm_score_v2.mean().round(2).to_dict())

merged.to_csv(f"{SAVE_DIR}/annotated_100.csv", index=False)
print("\nSaved to", SAVE_DIR)


Pairs scored: 100 of 100
AI score counts (v2): {0: 67, 1: 20, 2: 4, 3: 9}

AI v2 vs my scores: Spearman = 62.50
AI v2 vs levels:    Spearman = 68.20
AI v2 average per level: {0: 0.0, 1: 0.08, 2: 0.56, 3: 1.56}

Saved to /content/data
